In [73]:
import pandas as pd

In [83]:
unchar = pd.read_csv("../data/raw/uncharacterized_proteins.csv")
train = pd.read_csv("../data/raw/train_data.csv")

C:\Users\jclle\AppData\Local\Temp\ipykernel_9612\596741881.py:2: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/raw/train_data.csv")


In [94]:
# 'uniprotAccession'은 예외로 남기기
cols_to_drop = [col for col in unchar.columns if col in train.columns and col != "uniprotAccession"]

# 제거
unchar1 = unchar.drop(columns=cols_to_drop, errors='ignore')

# 확인
print("❌ 제거한 컬럼들:", cols_to_drop)
print("✅ 남은 unchar1 컬럼들:", unchar1.columns.tolist())

❌ 제거한 컬럼들: ['entryId', 'sequenceChecksum', 'sequenceVersionDate', 'uniprotId', 'uniprotDescription', 'organismScientificName', 'globalMetricValue', 'uniprotStart', 'uniprotEnd', 'uniprotSequence', 'modelCreatedDate', 'proteinFullNames', 'latestVersion', 'allVersions', '_version_', 'proteinShortNames']
✅ 남은 unchar1 컬럼들: ['gene', 'geneSynonyms', 'isReferenceProteome', 'isReviewed', 'uniprotAccession', 'taxId', 'organismCommonNames', 'isAMdata', 'organismScientificNameT']


In [95]:
# 전체 컬럼 리스트
cols = ['gene', 'geneSynonyms', 'isReferenceProteome', 'isReviewed', 
        'uniprotAccession', 'taxId', 'organismCommonNames', 
        'isAMdata', 'organismScientificNameT']

# 남기고 싶은 컬럼만 지정
keep_cols = ['gene', 'uniprotAccession']

# 제거할 컬럼 계산
drop_cols = [col for col in cols if col not in keep_cols]

print("❌ 삭제할 컬럼들:", drop_cols)

unchar1 = unchar1.drop(columns=drop_cols, errors='ignore')

❌ 삭제할 컬럼들: ['geneSynonyms', 'isReferenceProteome', 'isReviewed', 'taxId', 'organismCommonNames', 'isAMdata', 'organismScientificNameT']


In [97]:
unchar1 = unchar1.drop(columns=drop_cols, errors='ignore')
unchar1.head()

,gene,uniprotAccession
0,TP53TG5,Q9Y2B4
1,TP53TG3,Q9ULZ0
2,TCTA,P57738
3,PACRG,Q96M98
4,EBI3,Q14213


In [99]:
def remove_columns_same_as_train(train, unchar1):
    drop_cols = []
    
    for col_unchar in unchar1.columns:
        if col_unchar in train.columns:
            if train[col_unchar].equals(unchar1[col_unchar]):
                drop_cols.append(col_unchar)
    
    return unchar1.drop(columns=drop_cols)

df_cleaned = remove_columns_same_as_train(train, unchar1)

df_cleaned.shape

(10036, 2)

In [132]:
# 1️⃣ 조인 키 컬럼이 train에 없으면 예외 처리
if 'uniprotAccession' not in train.columns:
    raise KeyError("train에 'uniprotAccession' 컬럼이 없습니다. 조인 불가능!")

# 2️⃣ unchar1에 gene 컬럼이 없으면 병합 후에도 gene이 추가되지 않음
if 'gene' not in unchar1.columns:
    print("⚠️ 주의: unchar1에 'gene' 컬럼이 없어서 병합해도 추가되지 않음")

# 3️⃣ left join 수행
df_merged = train.merge(unchar1, on='uniprotAccession', how='left')

# 4️⃣ 확인
print("✅ 병합 결과 shape:", df_merged.shape)
print(df_merged[['uniprotAccession', 'gene']].head())

df_merged.head()
df_merged.shape

✅ 병합 결과 shape: (340156, 38)
  uniprotAccession    gene
0           Q0VGE8  ZNF816
1           Q0VGE8  ZNF816
2           Q0VGE8  ZNF816
3           Q0VGE8  ZNF816
4           Q0VGE8  ZNF816


(340156, 38)

In [133]:
# index컬럼 삭제하기
if 'index' in df_merged.columns:
    df_merged = df_merged.drop(columns='index')

In [134]:
# ✅ 1. 중복 행 제거
df_clean = df_merged.drop_duplicates(keep='first')

# ✅ 2. 중복 컬럼 제거 함수 (제거된 컬럼 로그까지)
def remove_duplicate_value_columns(df):
    duplicate_cols = set()
    cols = df.columns.tolist()

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            col1 = cols[i]
            col2 = cols[j]
            if col2 not in duplicate_cols:
                if df[col1].equals(df[col2]):
                    duplicate_cols.add(col2)
                    print(f"🧹 중복 컬럼 제거: '{col2}' (≡ '{col1}')")

    return df.drop(columns=list(duplicate_cols)), list(duplicate_cols)

# ✅ 3. 중복 컬럼 제거 적용
df_cleaned, removed_cols = remove_duplicate_value_columns(df_clean)

# ✅ 4. 중복 행 (이미 제거된 것) 다시 추출해서 보여주기
duplicates = df_merged[df_merged.duplicated(keep='first')]

print("\n====================================")
print("✅ 최종 제거된 중복 컬럼 수:", len(removed_cols))
print("🧹 제거된 중복 컬럼 목록:", removed_cols)

print("\n❌ 제거된 중복 행 수:", len(duplicates))
print("🧾 중복 행 샘플:")
print(duplicates.head())
print("====================================")


🧹 중복 컬럼 제거: 'uniprotAccession_unchar' (≡ 'UniProt_ID')
🧹 중복 컬럼 제거: 'gene' (≡ 'gene_x')

✅ 최종 제거된 중복 컬럼 수: 2
🧹 제거된 중복 컬럼 목록: ['uniprotAccession_unchar', 'gene']

❌ 제거된 중복 행 수: 0
🧾 중복 행 샘플:
Empty DataFrame
Columns: [Disease ID, Disease Name, Gene ID, UniProt_ID, GO_Terms, PDB_IDs, PubMed_IDs, protein1, protein2, combined_score, entryId, gene_x, sequenceChecksum, sequenceVersionDate, uniprotAccession, uniprotId, uniprotDescription, organismScientificName, globalMetricValue, uniprotStart, uniprotEnd, uniprotSequence, modelCreatedDate, proteinFullNames, latestVersion, allVersions, _version_, proteinShortNames, uniprotAccession_unchar, entry_name, protein_name, organism, gene_y, protein_existence, sequence_version, sequence, gene]
Index: []

[0 rows x 37 columns]


In [123]:
# 동일한 값 가진 컬럼들 제거 (첫 번째(= train_data 컬럼을 기준) 컬럼만 남김)
df_clean = df_merged.drop_duplicates(keep='first') 

def remove_duplicate_value_columns(df_clean):
    duplicate_cols = set()
    cols = df_clean.columns.tolist()
    
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            col1 = cols[i]
            col2 = cols[j]
            if col2 not in duplicate_cols:
                if df_clean[col1].equals(df_clean[col2]):
                    duplicate_cols.add(col2)
    
    return df_clean.drop(columns=list(duplicate_cols))

# df_cleaned가 중복 컬럼까지 제거한 변수
df_cleaned = remove_duplicate_value_columns(df_clean)
df_cleaned.head()

# 중복된 행만 따로 보기 (전체 컬럼 기준으로 완전히 같은 행)
duplicates1 = df_clean[df_clean.duplicated(keep='first')]
print("❌ 제거된 중복 행 수:", len(duplicates1))
print("🧹 제거된 중복 행 미리보기:")

❌ 제거된 중복 행 수: 0
🧹 제거된 중복 행 미리보기:


In [ ]:
# y가 모든 유전자 값을 가지고 있어서 gene_x, gene을 제거해야한다는 것을 알았다.
print(df_cleaned['gene_y'].count())

340156


In [137]:
df_cleaned.to_csv("../data/merged_data.csv", index=False)

In [136]:
df_cleaned.columns

Index(['Disease ID', 'Disease Name', 'Gene ID', 'UniProt_ID', 'GO_Terms',
       'PDB_IDs', 'PubMed_IDs', 'protein1', 'protein2', 'combined_score',
       'entryId', 'gene_x', 'sequenceChecksum', 'sequenceVersionDate',
       'uniprotAccession', 'uniprotId', 'uniprotDescription',
       'organismScientificName', 'globalMetricValue', 'uniprotStart',
       'uniprotEnd', 'uniprotSequence', 'modelCreatedDate', 'proteinFullNames',
       'latestVersion', 'allVersions', '_version_', 'proteinShortNames',
       'entry_name', 'protein_name', 'organism', 'gene_y', 'protein_existence',
       'sequence_version', 'sequence'],
      dtype='object')

미지 단백질 10프로 추출

In [139]:
merged_data = pd.read_csv("../data/merged_data.csv")

C:\Users\jclle\AppData\Local\Temp\ipykernel_9612\1180975300.py:1: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_data = pd.read_csv("../data/merged_data.csv")


In [143]:
# 10% 랜덤 샘플 추출 (seed 고정 시 reproducible)
merged_data_small = merged_data.sample(frac=0.1, random_state=42)

# 파일로 저장 (CSV 포맷, 인덱스는 저장하지 않음)
merged_data_small.to_csv("../data/merged_data_small.csv", index=False)

print("✅ 10% 샘플 추출 완료! → merged_data_small.csv")

✅ 10% 샘플 추출 완료! → merged_data_small.csv
